# Лабораторная работа №1
## ML-пайплайн интеллектуальной аудитории: определение занятости помещения

**Курс:** Технологии машинного обучения
**Направление:** 44.04.01 Педагогическое образование — «Умные системы и интернет вещей в образовании»
**Раздел 1.** Понятие машинного обучения · **20 баллов БРС**

---

### Инженерно-педагогическая постановка

В кабинете установлена система «умная аудитория». Контроллер вентиляции, света и отопления должен ответить на один вопрос: **есть ли сейчас в классе люди?**

Камеру на потолок повесить нельзя:
* видеопоток из класса с несовершеннолетними — биометрические персональные данные особой категории (152-ФЗ);
* постоянное наблюдение разрушает доверительную образовательную среду;
* стоимость решения на порядок выше комплекта датчиков среды.

Датчик CO₂ не «видит» человека — он видит только интегральный след его дыхания. Это **приватность по построению**: из показаний физически невозможно восстановить личность. Ваша задача — экспериментально проверить, достаточно ли такого «слепого» сигнала для управления автоматикой класса, и **честно** оценить цену ошибок системы.

### Что нужно сделать

| Блок | Действие |
|---|---|
| 1 | Загрузить UCI Occupancy Detection (ID 357), выделить подвыборку своего варианта |
| 2 | Провести EDA: баланс классов, распределения, корреляции — **с письменными выводами** |
| 3 | Найти и устранить **три** источника утечки данных |
| 4 | Собрать `Pipeline`, обучить baseline + модель варианта + 2 модели сравнения |
| 5 | Посчитать Confusion matrix, Precision, Recall, F1, ROC-AUC |
| 6 | Подобрать порог под приоритетную метрику варианта |
| 7 | Написать инженерно-педагогическую интерпретацию ошибок |

**Семь блоков `# TODO: СТУДЕНТ` и шесть ячеек `TEST` — все шесть тестов должны выдать `✅ PASSED`.**

## Блок 0. Теоретическое введение

### 0.1. Формализация задачи обучения с учителем

Дана выборка $D = \{(x_i, y_i)\}_{i=1}^{N}$, где $x_i \in X \subseteq \mathbb{R}^d$ — вектор показаний датчиков в момент $i$, а $y_i \in Y = \{0, 1\}$ — метка занятости помещения.

Требуется построить решающую функцию

$$f_\theta: X \rightarrow Y,$$

параметризованную вектором $\theta$. Само по себе «машинное обучение» — это процедура выбора $\theta$ по данным, а не по инструкции программиста. В этом принципиальное отличие от классической автоматики: инженер не пишет правило «если CO₂ > 800, то занято», а задаёт **семейство** правил и **критерий** их качества.

### 0.2. Эмпирический риск

Идеальный критерий — **истинный риск** (математическое ожидание потерь по неизвестному распределению $P(x, y)$):

$$R(\theta) = \mathbb{E}_{(x,y) \sim P}\big[L(f_\theta(x), y)\big].$$

Распределение $P$ неизвестно, поэтому минимизируется его выборочная оценка — **эмпирический риск**:

$$\theta^* = \arg\min_{\theta} \; \frac{1}{N}\sum_{i=1}^{N} L\big(f_\theta(x_i),\, y_i\big) \; + \; \lambda\,\Omega(\theta).$$

Для логистической регрессии $L$ — бинарная кросс-энтропия:

$$L(\hat{p}, y) = -\big[y \log \hat{p} + (1 - y)\log(1 - \hat{p})\big], \qquad \hat{p} = \sigma(w^\top x + b) = \frac{1}{1 + e^{-(w^\top x + b)}}.$$

Слагаемое $\lambda\,\Omega(\theta)$ — **регуляризатор**, штраф за сложность модели (в sklearn для `LogisticRegression` по умолчанию $\Omega(\theta) = \|w\|_2^2$, а сила штрафа задаётся параметром `C` = $1/\lambda$).

### 0.3. Переобучение

**Переобучение (overfitting)** — ситуация, когда $\frac{1}{N}\sum L \rightarrow 0$ на обучающей выборке, но $R(\theta)$ на новых данных остаётся большим. Модель выучила шум и частности конкретной выборки вместо закономерности.

$$\underbrace{R(\theta)}_{\text{истинная ошибка}} \approx \underbrace{\hat{R}_{train}(\theta)}_{\text{ошибка на train}} + \underbrace{\text{gap}}_{\text{разрыв обобщения}}$$

Разрыв растёт с ростом ёмкости модели и падает с ростом объёма данных. Три практических инструмента контроля: **отложенная выборка**, **регуляризация**, **ограничение ёмкости** (например, `max_depth` у дерева).

Отдельно от переобучения стоит **утечка данных (data leakage)** — попадание в обучение информации, недоступной модели в момент реального предсказания. Внешне утечка выглядит как выдающееся качество, а на практике даёт полный провал. В этой работе вам предстоит устранить три утечки — см. блок 3.

### 0.4. Метрики бинарной классификации

Обозначим элементы **матрицы ошибок (confusion matrix)**:

| | Предсказано 0 (пусто) | Предсказано 1 (занято) |
|---|---|---|
| **Факт 0 (пусто)** | TN | **FP** — ложная тревога |
| **Факт 1 (занято)** | **FN** — пропуск | TP |

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}, \qquad
\text{Precision} = \frac{TP}{TP + FP}, \qquad
\text{Recall} = \frac{TP}{TP + FN},$$

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}, \qquad
F_\beta = (1 + \beta^2)\frac{\text{Precision} \cdot \text{Recall}}{\beta^2\,\text{Precision} + \text{Recall}}.$$

**Почему accuracy здесь бесполезна.** Если аудитория пуста 78 % времени, тривиальный классификатор «всегда пусто» даёт accuracy = 0.78, Precision = 0, Recall = 0 и нулевую практическую ценность. Поэтому в работе обязателен **baseline** (`DummyClassifier`) — точка отсчёта, ниже которой опускаться нельзя.

**Инженерный смысл в терминах умной школы:**
* **FP** (сказали «занято», а класс пуст) → вентиляция и свет работают вхолостую → перерасход электроэнергии, шум, сквозняк.
* **FN** (сказали «пусто», а в классе 30 детей) → вентиляция выключена → CO₂ растёт до 1500–2000 ppm → головная боль, падение концентрации внимания, снижение результатов контрольных.

Это **асимметричные** ошибки: цена FN в образовательной среде выше цены FP. Отсюда — приоритет **Recall** в большинстве вариантов. Но приоритет Precision тоже встречается: например, если вентиляция шумная и мешает вести урок, а бюджет школы на электроэнергию жёстко лимитирован.

## Блок 1. Настройка окружения и конфигурация варианта

In [ ]:
# Установка зависимостей (в Google Colab выполняется один раз за сессию).
# ucimlrepo — официальный клиент репозитория UCI.
!pip install -q ucimlrepo scikit-learn pandas matplotlib seaborn

In [ ]:
import sys, warnings, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

warnings.filterwarnings("ignore")

# --- ВОСПРОИЗВОДИМОСТЬ (критерий 7, 1 балл) -----------------------------
SEED = 42
np.random.seed(SEED)

# --- Оформление графиков ------------------------------------------------
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

print("Python      :", sys.version.split()[0], "|", platform.system())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("SEED        :", SEED)

In [ ]:
# ======================================================================
#  УКАЖИТЕ ЗДЕСЬ СВОЙ НОМЕР ВАРИАНТА ИЗ ВЕДОМОСТИ (1..25)
# ======================================================================
VARIANT = 1   # <-- ИЗМЕНИТЕ НА СВОЙ НОМЕР
# ======================================================================

In [ ]:
# Справочники параметризации (соответствуют разделу 5.1 Readme_lab_01.md).
IMPUTERS = ["mean", "median", "most_frequent", "knn"]

SENSOR_SETS = {
    "S1": ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"],
    "S2": ["Temperature", "Humidity", "CO2"],
    "S3": ["Temperature", "Light", "CO2"],
    "S4": ["Humidity", "Light", "CO2", "HumidityRatio"],
    "S5": ["Temperature", "Humidity", "Light", "CO2"],
}
SENSOR_ORDER = ["S1", "S2", "S3", "S4", "S5"]

MODELS = ["LogisticRegression", "DecisionTreeClassifier",
          "GaussianNB", "KNeighborsClassifier"]


def get_variant_config(v: int) -> dict:
    """Возвращает полную конфигурацию варианта v по правилам раздела 5.1.

    missing_rate  = 0.01 + 0.005 * (v mod 5)
    imputer       = IMPUTERS[(v-1) mod 4]
    sensors       = SENSOR_SETS[SENSOR_ORDER[(v-1) mod 5]]
    model         = MODELS[(v-1) mod 4]
    priority      = Recall для нечётных v, Precision для чётных
    """
    assert isinstance(v, int) and 1 <= v <= 25, "VARIANT должен быть целым числом 1..25"
    sid = SENSOR_ORDER[(v - 1) % 5]
    return {
        "variant": v,
        "missing_rate": round(0.01 + 0.005 * (v % 5), 4),
        "imputer": IMPUTERS[(v - 1) % 4],
        "sensor_set_id": sid,
        "sensors": SENSOR_SETS[sid],
        "model": MODELS[(v - 1) % 4],
        "priority_metric": "Recall" if v % 2 == 1 else "Precision",
        "random_state": SEED + v,
    }


CFG = get_variant_config(VARIANT)

print("=" * 62)
print(f"  КОНФИГУРАЦИЯ ВАРИАНТА №{CFG['variant']}")
print("=" * 62)
print(f"  Доля пропусков (missing_rate) : {CFG['missing_rate']:.3f}  "
      f"[= 0.01 + 0.005 * ({VARIANT} mod 5 = {VARIANT % 5})]")
print(f"  Метод импутации               : {CFG['imputer']}")
print(f"  Набор сенсоров                : {CFG['sensor_set_id']} -> {CFG['sensors']}")
print(f"  Основная модель               : {CFG['model']}")
print(f"  Приоритетная метрика          : {CFG['priority_metric']}")
print(f"  random_state варианта         : {CFG['random_state']}")
print("=" * 62)

### 1.1. Загрузка данных

Реализованы три пути с автоматическим переключением:

1. **`ucimlrepo`** — `fetch_ucirepo(id=357)`, рекомендуемый способ;
2. **прямой ZIP-архив** с `archive.ics.uci.edu`;
3. **синтетический генератор-фолбэк** — включается, только если оба сетевых пути недоступны.

> Фолбэк воспроизводит физику помещения (тепловыделение человека ≈ 100 Вт, выдох ≈ 40 000 ppm CO₂, инерция накопления CO₂) и нужен исключительно для отладки кода. **Если сработал фолбэк — обязательно укажите это в выводах**; переменная `DATA_SOURCE` хранит фактический источник.

In [ ]:
import io, zipfile, urllib.request

UCI_ZIP = "https://archive.ics.uci.edu/static/public/357/occupancy+detection.zip"
EXPECTED_COLS = ["date", "Temperature", "Humidity", "Light",
                 "CO2", "HumidityRatio", "Occupancy"]


def _humidity_ratio(t_c, rh_pct, p_kpa=101.325):
    """Абсолютная влажность (кг влаги / кг сухого воздуха) по формуле Магнуса."""
    psat = 0.61078 * np.exp((17.2694 * t_c) / (t_c + 237.3))   # кПа
    pv = (rh_pct / 100.0) * psat
    return 0.622 * pv / np.maximum(p_kpa - pv, 1e-6)


def make_synthetic_occupancy(n=20560, seed=SEED):
    """Физически мотивированный генератор-фолбэк (см. пояснение выше)."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2015-02-02 14:19:00", periods=n, freq="min")
    minute = idx.hour * 60 + idx.minute
    weekday = idx.dayofweek.values

    work = ((minute >= 8 * 60 + 30) & (minute <= 18 * 60) & (weekday < 5)).astype(float)
    lunch = ((minute >= 12 * 60 + 30) & (minute <= 13 * 60 + 15)).astype(float)
    p = work * (1 - 0.75 * lunch)

    occ, state = np.zeros(n, dtype=int), 0
    for i in range(n):                       # марковская цепь занятости
        p_on, p_off = 0.010 + 0.22 * p[i], 0.004 + 0.30 * (1 - p[i])
        if state == 0 and rng.random() < p_on:
            state = 1
        elif state == 1 and rng.random() < p_off:
            state = 0
        occ[i] = state

    kernel = np.exp(-np.arange(90) / 25.0); kernel /= kernel.sum()
    occ_smooth = np.convolve(occ, kernel, mode="full")[:n]     # инерция среды
    day_wave = 0.9 * np.sin(2 * np.pi * (minute - 300) / 1440.0)

    temp = 19.6 + day_wave + 1.55 * occ_smooth + rng.normal(0, 0.09, n)
    hum = np.clip(26.0 + 0.9 * np.sin(2 * np.pi * (minute - 600) / 1440.0)
                  + 2.4 * occ_smooth + rng.normal(0, 0.35, n), 15, 45)

    forgotten = (rng.random(n) < 0.020) & (occ == 0)   # свет забыли выключить
    dark_class = (rng.random(n) < 0.045) & (occ == 1)  # урок с проектором, свет выключен
    lamp = np.clip(occ + forgotten - dark_class, 0, 1)
    light = np.clip(lamp * rng.normal(450, 45, n) + (1 - lamp) * rng.normal(2.0, 1.5, n), 0, None)

    co2 = 440 + 620 * occ_smooth + rng.normal(0, 18, n)

    return pd.DataFrame({
        "date": idx,
        "Temperature": np.round(temp, 3),
        "Humidity": np.round(hum, 4),
        "Light": np.round(light, 2),
        "CO2": np.round(co2, 2),
        "HumidityRatio": np.round(_humidity_ratio(temp, hum), 8),
        "Occupancy": occ,
    })


def load_occupancy(verbose=True):
    """Возвращает (DataFrame, source) — источник: 'ucimlrepo' | 'uci_zip' | 'synthetic'."""
    # --- Путь 1: ucimlrepo -------------------------------------------------
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=357)
        X, y = ds.data.features.copy(), ds.data.targets.copy()
        df = pd.concat([X, y], axis=1)
        df.columns = [c.strip() for c in df.columns]
        if "Occupancy" not in df.columns:
            df = df.rename(columns={df.columns[-1]: "Occupancy"})
        if verbose:
            print("[OK] Источник: ucimlrepo (fetch_ucirepo(id=357))")
        return df, "ucimlrepo"
    except Exception as e:
        if verbose:
            print(f"[!] ucimlrepo недоступен: {type(e).__name__}: {e}")

    # --- Путь 2: прямой ZIP ------------------------------------------------
    try:
        with urllib.request.urlopen(UCI_ZIP, timeout=30) as r:
            blob = r.read()
        frames = []
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            for name in sorted(z.namelist()):
                if name.endswith(".txt"):
                    with z.open(name) as fh:
                        frames.append(pd.read_csv(fh))
        df = pd.concat(frames, ignore_index=True)
        df.columns = [c.strip().strip('"') for c in df.columns]
        if verbose:
            print("[OK] Источник: прямой ZIP archive.ics.uci.edu")
        return df, "uci_zip"
    except Exception as e:
        if verbose:
            print(f"[!] Прямая загрузка недоступна: {type(e).__name__}: {e}")

    # --- Путь 3: фолбэк ----------------------------------------------------
    if verbose:
        print("[i] Включён СИНТЕТИЧЕСКИЙ ФОЛБЭК — укажите это в выводах!")
    return make_synthetic_occupancy(), "synthetic"


df_raw, DATA_SOURCE = load_occupancy()

df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw = df_raw.sort_values("date").reset_index(drop=True)
df_raw["Occupancy"] = df_raw["Occupancy"].astype(int)

print(f"\nРазмер полного датасета : {df_raw.shape}")
print(f"Период наблюдений      : {df_raw['date'].min()} — {df_raw['date'].max()}")
print(f"Шаг дискретизации      : {df_raw['date'].diff().mode()[0]}")
df_raw.head()

In [ ]:
# Подвыборка варианта: непрерывное окно 60 % ряда, сдвинутое по номеру варианта.
# Именно непрерывное, а не случайное, — иначе разрушится временная структура
# и станет невозможен корректный хронологический сплит (см. блок 3).

def variant_subsample(df: pd.DataFrame, variant: int, frac: float = 0.60) -> pd.DataFrame:
    n = len(df)
    win = int(frac * n)
    start = int(round((n - win) * (variant - 1) / 24))
    return df.iloc[start:start + win].reset_index(drop=True)


df_v = variant_subsample(df_raw, VARIANT)

print(f"Вариант {VARIANT}: подвыборка {df_v.shape[0]} строк "
      f"({df_v['date'].min()} — {df_v['date'].max()})")
print(f"Пропущенных значений в исходных данных: {int(df_v.isna().sum().sum())}")
df_v.describe().T.round(3)

## Блок 2. Разведочный анализ данных (EDA) — 4 балла

> **Требование к оценке.** Каждый график ниже должен сопровождаться вашим **письменным выводом** в отдельной markdown-ячейке. Формулировки вида «график построен» не засчитываются. Ожидается: что видно, чем объясняется физически, какие следствия для модели.

In [ ]:
# 2.1. Баланс классов — числа и график.
counts = df_v["Occupancy"].value_counts().sort_index()
shares = df_v["Occupancy"].value_counts(normalize=True).sort_index()

print("Баланс классов в подвыборке варианта:")
for k in counts.index:
    label = "пусто (0)" if k == 0 else "занято (1)"
    print(f"  {label:12s}: {counts[k]:6d} наблюдений ({shares[k]*100:5.2f} %)")

majority_share = shares.max()
print(f"\nДоля мажоритарного класса = {majority_share:.4f}")
print(f"=> accuracy тривиального классификатора 'всегда мажоритарный' = {majority_share:.4f}")
print("   ЛЮБАЯ модель с accuracy ниже этого числа бесполезна.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(["пусто (0)", "занято (1)"], counts.values, color=["#4C72B0", "#C44E52"])
ax[0].set_title(f"Баланс классов, вариант {VARIANT}")
ax[0].set_xlabel("Класс Occupancy"); ax[0].set_ylabel("Число наблюдений")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f"{v}\n{shares.values[i]*100:.1f}%", ha="center", va="bottom", fontsize=9)

ax[1].plot(df_v["date"], df_v["Occupancy"], lw=0.6, color="#C44E52")
ax[1].set_title("Занятость во времени (структура ряда)")
ax[1].set_xlabel("Время"); ax[1].set_ylabel("Occupancy")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

**ВЫВОД 2.1 (заполните):**
_Опишите: насколько выражен дисбаланс; какой accuracy даст тривиальный классификатор; видна ли на правом графике суточная/недельная периодичность; почему из-за этой периодичности нельзя перемешивать данные при сплите._

In [ ]:
# 2.2. Корреляционная матрица (heatmap) — только доступные по варианту сенсоры + цель.
cols_corr = CFG["sensors"] + ["Occupancy"]
corr = df_v[cols_corr].corr()

fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_corr))); ax.set_xticklabels(cols_corr, rotation=45, ha="right")
ax.set_yticks(range(len(cols_corr))); ax.set_yticklabels(cols_corr)
for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.values[i, j]) > 0.55 else "black", fontsize=9)
ax.set_title(f"Корреляции Пирсона, набор {CFG['sensor_set_id']} (вариант {VARIANT})")
fig.colorbar(im, ax=ax, shrink=0.8, label="коэффициент корреляции")
plt.tight_layout(); plt.show()

print("Корреляция признаков с целевой переменной (по убыванию |r|):")
print(corr["Occupancy"].drop("Occupancy").abs().sort_values(ascending=False).round(3))

**ВЫВОД 2.2 (заполните):**
_Какой сенсор сильнее всего связан с занятостью? Есть ли пары сильно скоррелированных признаков (мультиколлинеарность) и чем она грозит логистической регрессии? Если в вашем наборе есть `Light` — объясните, почему высокая корреляция здесь не повод радоваться._

In [ ]:
# 2.3. Распределения признаков в разрезе классов.
sensors = CFG["sensors"]
ncols = min(3, len(sensors))
nrows = int(np.ceil(len(sensors) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows))
axes = np.atleast_1d(axes).ravel()

for k, col in enumerate(sensors):
    a = axes[k]
    a.hist(df_v.loc[df_v.Occupancy == 0, col], bins=45, alpha=0.65,
           label="пусто (0)", color="#4C72B0", density=True)
    a.hist(df_v.loc[df_v.Occupancy == 1, col], bins=45, alpha=0.65,
           label="занято (1)", color="#C44E52", density=True)
    a.set_title(col); a.set_xlabel(col); a.set_ylabel("плотность"); a.legend(fontsize=8)

for k in range(len(sensors), len(axes)):
    axes[k].axis("off")
fig.suptitle(f"Распределения признаков по классам, вариант {VARIANT}", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# TODO: СТУДЕНТ (EDA, дополнение к критерию 2)
# Постройте boxplot каждого доступного сенсора в разрезе классов
# и выведите таблицу групповых средних df_v.groupby('Occupancy')[sensors].mean().
# Подсказка: plt.subplots + ax.boxplot([data0, data1], tick_labels=['0','1'])
# Не забудьте заголовок, подписи осей и легенду — иначе штраф −1 балл за график.

# ВАШ КОД ЗДЕСЬ

**ВЫВОД 2.3 (заполните):**
_Для каких признаков распределения классов разделяются, а для каких перекрываются? Насколько в среднем растёт CO₂ при появлении людей? Согласуется ли это с физикой (выдох ≈ 40 000 ppm против фона 400 ppm)?_

## Блок 3. Data Leakage и хронологический сплит — 4 балла

В этой работе спрятаны **три** утечки. Найти и устранить нужно все три.

**Утечка №1 — временна́я метка `date`.**
Столбец `date` содержит дату и время. Модель, получив его на вход, выучит не физику помещения, а **расписание конкретного офиса**: «в понедельник с 9:00 занято». Accuracy будет почти 1.0, а при переносе в другую школу (другой часовой пояс, другое расписание, каникулы) модель развалится. **`date` удаляем из `X`**, но сохраняем отдельно для сортировки и графиков.

**Утечка №2 — случайное разбиение временного ряда.**
Замеры идут с шагом 1 минута; соседние точки почти идентичны (автокорреляция > 0.99). `train_test_split(shuffle=True)` разбросает соседние минуты одного урока между train и test — модель фактически увидит ответы. Метрики завышаются на 5–15 п. п. **Нужен хронологический сплит:** первые 70 % ряда — train, последние 30 % — test.

**Утечка №3 — препроцессинг до сплита.**
`StandardScaler.fit()` или `SimpleImputer.fit()` на объединённой выборке переносят в обучение статистики теста (среднее, дисперсию, медиану). **Все преобразования — только внутри `Pipeline`**, `fit` — только на `X_train`.

In [ ]:
# 3.1. Формируем X и y. Столбец 'date' в X НЕ ПОПАДАЕТ (устранение утечки №1).
timestamps = df_v["date"].copy()          # сохраняем отдельно: для графиков, не для обучения
X_clean = df_v[CFG["sensors"]].copy()     # только сенсоры своего варианта
y = df_v["Occupancy"].copy()

assert "date" not in X_clean.columns, "Утечка №1: столбец date остался в X!"
print("Признаки в X :", list(X_clean.columns))
print("Форма X      :", X_clean.shape, "| форма y:", y.shape)

In [ ]:
# 3.2. Внесение искусственных пропусков по формуле варианта (механизм MCAR).
# Пропуски вносятся ТОЛЬКО в матрицу признаков; целевая переменная не портится.

def inject_missing(X: pd.DataFrame, rate: float, random_state: int) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    Xm = X.copy().astype(float)
    mask = pd.DataFrame(rng.random(Xm.shape) < rate, index=Xm.index, columns=Xm.columns)
    return Xm.mask(mask)


X = inject_missing(X_clean, CFG["missing_rate"], CFG["random_state"])

print(f"Заданная доля пропусков (вариант {VARIANT}): {CFG['missing_rate']:.3f}")
print(f"Фактическая доля пропусков              : {X.isna().mean().mean():.4f}")
print(f"Всего NaN                               : {int(X.isna().sum().sum())}")
print("\nПропуски по столбцам:")
print(X.isna().sum().to_string())

In [ ]:
# TODO-1: СТУДЕНТ — ХРОНОЛОГИЧЕСКИЙ СПЛИТ (устранение утечки №2)
#
# Реализуйте функцию chronological_split: первые (1 - test_size) долей ряда -> train,
# оставшиеся -> test. НИКАКОГО перемешивания.
# Требования:
#   * порядок строк не меняется;
#   * cut = int(round(len(X) * (1 - test_size)));
#   * возвращать X_train, X_test, y_train, y_test с сохранёнными индексами (.iloc).

def chronological_split(X: pd.DataFrame, y: pd.Series, test_size: float = 0.30):
    """Разбиение временного ряда по времени, без перемешивания."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте chronological_split")


X_train, X_test, y_train, y_test = chronological_split(X, y, test_size=0.30)

print(f"train: {X_train.shape[0]} строк | доля класса 1 = {y_train.mean():.3f}")
print(f"test : {X_test.shape[0]} строк | доля класса 1 = {y_test.mean():.3f}")
print(f"train период: {timestamps.iloc[X_train.index].min()} — {timestamps.iloc[X_train.index].max()}")
print(f"test  период: {timestamps.iloc[X_test.index].min()} — {timestamps.iloc[X_test.index].max()}")

In [ ]:
# ====================== TEST 1 — корректность сплита ======================
# Проверка структурная: обученные модели ещё не нужны.
def test_1_split():
    assert len(X_train) + len(X_test) == len(X), "Потеряны или продублированы строки"
    assert abs(len(X_test) / len(X) - 0.30) < 0.01, "Доля теста должна быть ~30 %"
    assert set(X_train.index).isdisjoint(set(X_test.index)), "train и test пересекаются"
    assert X_train.index.max() < X_test.index.min(), \
        "Сплит НЕ хронологический: индексы train должны идти строго раньше test"
    assert list(X_train.index) == sorted(X_train.index), "Порядок train нарушен (было перемешивание?)"
    assert "date" not in X_train.columns, "Утечка №1: date в признаках"
    assert list(X_train.columns) == CFG["sensors"], "Набор сенсоров не соответствует варианту"
    assert y_train.shape[0] == X_train.shape[0] and y_test.shape[0] == X_test.shape[0]
    print("✅ TEST 1 PASSED — сплит хронологический, утечки №1 и №2 устранены")

test_1_split()

### Обоснование сплита (заполните — оценивается в критерии 3)

_Напишите 4–6 предложений: почему для этих данных случайное перемешивание недопустимо; чему равна автокорреляция соседних отсчётов; чем хронологический сплит ближе к реальной эксплуатации системы; какой ценой (какие риски появляются вместо утечки — например, сдвиг распределения между train и test)._

## Блок 4. Preprocessing, Baseline и модели — 4 балла

### Почему именно `Pipeline`

`sklearn.pipeline.Pipeline` — это не «красивая обёртка», а **техническое средство защиты от утечки №3**. Внутри пайплайна:

* `pipe.fit(X_train, y_train)` вызывает `fit_transform` препроцессоров **только на train**;
* `pipe.predict(X_test)` вызывает `transform` (без `fit`) с параметрами, выученными на train.

Порядок шагов в этой работе:

```
Imputer (заполнение NaN) → StandardScaler (масштабирование) → Model
```

Импутация **до** масштабирования: `StandardScaler` не умеет работать с `NaN`. Масштабирование обязательно для `LogisticRegression` (сходимость и корректная регуляризация) и для `KNeighborsClassifier` (евклидово расстояние). Для `DecisionTreeClassifier` и `GaussianNB` оно избыточно, но безвредно — единый пайплайн даёт честное сравнение моделей.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve,
                             ConfusionMatrixDisplay)

In [ ]:
# TODO-2: СТУДЕНТ — СБОРКА PIPELINE (устранение утечки №3)
#
# Реализуйте две фабрики.
#
# make_imputer(name):
#   'mean' / 'median' / 'most_frequent' -> SimpleImputer(strategy=name)
#   'knn'                               -> KNNImputer(n_neighbors=5)
#
# build_pipeline(model_name, imputer_name, random_state=SEED):
#   Pipeline([('imputer', ...), ('scaler', StandardScaler()), ('model', ...)])
#   Модели:
#     LogisticRegression(max_iter=1000, random_state=random_state)
#     DecisionTreeClassifier(max_depth=5, random_state=random_state)
#     GaussianNB()
#     KNeighborsClassifier(n_neighbors=7)
#   ВАЖНО: имена шагов ровно 'imputer', 'scaler', 'model' — их проверяет TEST 2.

def make_imputer(name: str):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте make_imputer")


def build_pipeline(model_name: str, imputer_name: str, random_state: int = SEED) -> Pipeline:
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError("Реализуйте build_pipeline")

In [ ]:
# ============ TEST 2 — структура пайплайна (без обучения, «на макетах») ============
def test_2_pipeline():
    p = build_pipeline(CFG["model"], CFG["imputer"])
    assert isinstance(p, Pipeline), "build_pipeline должна возвращать sklearn Pipeline"
    names = [n for n, _ in p.steps]
    assert names == ["imputer", "scaler", "model"], f"Ожидались шаги imputer/scaler/model, получено {names}"
    assert isinstance(p.named_steps["scaler"], StandardScaler), "Второй шаг — StandardScaler"
    imp = p.named_steps["imputer"]
    if CFG["imputer"] == "knn":
        assert isinstance(imp, KNNImputer) and imp.n_neighbors == 5, "Ожидался KNNImputer(n_neighbors=5)"
    else:
        assert isinstance(imp, SimpleImputer) and imp.strategy == CFG["imputer"], \
            f"Ожидался SimpleImputer(strategy='{CFG['imputer']}')"
    assert type(p.named_steps["model"]).__name__ == CFG["model"], \
        f"Модель варианта — {CFG['model']}"
    # Проверка на все четыре модели: фабрика должна быть универсальной
    for m in MODELS:
        assert type(build_pipeline(m, "mean").named_steps["model"]).__name__ == m
    print("✅ TEST 2 PASSED — Pipeline собран корректно, утечка №3 закрыта")

test_2_pipeline()

In [ ]:
# TODO-3: СТУДЕНТ — BASELINE
#
# Обучите DummyClassifier(strategy='most_frequent', random_state=SEED).
# ВНИМАНИЕ: DummyClassifier не умеет работать с NaN, поэтому оберните его
# в тот же Pipeline (imputer -> scaler -> dummy) — это ещё и честное сравнение.
# Сохраните обученный объект в переменную baseline и выведите его accuracy на тесте.

baseline = None
# ВАШ КОД ЗДЕСЬ

In [ ]:
# ====================== TEST 3 — baseline обучен ======================
def test_3_baseline():
    assert baseline is not None, "Переменная baseline не заполнена"
    pred = baseline.predict(X_test)
    assert len(np.unique(pred)) == 1, "most_frequent должен предсказывать один класс"
    acc = accuracy_score(y_test, pred)
    assert abs(acc - max(y_test.mean(), 1 - y_test.mean())) < 1e-6, \
        "Accuracy baseline должна равняться доле мажоритарного класса теста"
    print(f"✅ TEST 3 PASSED — baseline accuracy = {acc:.4f} (планка для всех моделей)")

test_3_baseline()

In [ ]:
# TODO-4: СТУДЕНТ — ОБУЧЕНИЕ МОДЕЛИ ВАРИАНТА
#
# 1. Соберите пайплайн модели своего варианта: build_pipeline(CFG['model'], CFG['imputer'])
# 2. Обучите на X_train, y_train.
# 3. Сохраните в переменную model_main.
# 4. Выведите classification_report(y_test, model_main.predict(X_test), digits=4).

model_main = None
# ВАШ КОД ЗДЕСЬ

In [ ]:
# TODO-5: СТУДЕНТ — ДВЕ МОДЕЛИ СРАВНЕНИЯ
#
# Выберите ЛЮБЫЕ ДВЕ модели из MODELS, отличные от модели вашего варианта,
# обучите их на тех же данных через build_pipeline и сложите ВСЕ обученные модели
# в словарь fitted = {'Baseline': baseline, CFG['model']: model_main, '<модель2>': ..., '<модель3>': ...}

fitted = {}
# ВАШ КОД ЗДЕСЬ

In [ ]:
# 4.1. Единая функция оценки (использует уже обученные модели — «живой» прогон).
def evaluate(name: str, estimator, X_te, y_te) -> dict:
    """Считает полный набор метрик для обученной модели."""
    y_pred = estimator.predict(X_te)
    try:
        y_prob = estimator.predict_proba(X_te)[:, 1]
    except (AttributeError, IndexError):
        y_prob = y_pred.astype(float)
    return {
        "Модель": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, y_prob),
        "PR-AUC": average_precision_score(y_te, y_prob),
    }


results = pd.DataFrame([evaluate(n, m, X_test, y_test) for n, m in fitted.items()])
results = results.set_index("Модель").round(4)
results

In [ ]:
# ============ TEST 4 — модели обучены и превосходят baseline ============
def test_4_models():
    assert len(fitted) >= 4, "Нужны baseline + модель варианта + 2 модели сравнения"
    assert "Baseline" in fitted, "В fitted нет ключа 'Baseline'"
    assert CFG["model"] in fitted, f"В fitted нет модели варианта {CFG['model']}"
    base_f1 = results.loc["Baseline", "F1"]
    best_f1 = results.drop(index="Baseline")["F1"].max()
    assert best_f1 > base_f1, "Ни одна модель не превзошла baseline по F1 — ищите ошибку"
    assert results.loc[CFG["model"], "F1"] > 0.5, \
        f"F1 модели варианта слишком низкая ({results.loc[CFG['model'], 'F1']:.3f})"
    assert results.drop(index="Baseline")["Accuracy"].max() < 0.9999, \
        "Accuracy ≈ 1.0 — почти наверняка осталась утечка данных"
    print(f"✅ TEST 4 PASSED — лучший F1 = {best_f1:.4f} против baseline F1 = {base_f1:.4f}")

test_4_models()

## Блок 5. Матрица ошибок и метрики — 3 балла

In [ ]:
# TODO-6: СТУДЕНТ — МАТРИЦЫ ОШИБОК
#
# Постройте confusion matrix для КАЖДОЙ модели из fitted (сетка subplots).
# Для модели своего варианта дополнительно распечатайте TN, FP, FN, TP числами:
#   tn, fp, fn, tp = confusion_matrix(y_test, model_main.predict(X_test)).ravel()
# Сохраните эти четыре числа в переменные tn, fp, fn, tp — их проверяет TEST 5.
# Подсказка: ConfusionMatrixDisplay.from_estimator(est, X_test, y_test, ax=ax, colorbar=False)

tn = fp = fn = tp = None
# ВАШ КОД ЗДЕСЬ

In [ ]:
# ====================== TEST 5 — матрица ошибок ======================
def test_5_confusion():
    assert None not in (tn, fp, fn, tp), "Заполните tn, fp, fn, tp"
    assert tn + fp + fn + tp == len(y_test), "Сумма элементов матрицы ≠ размеру теста"
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    assert abs(prec - results.loc[CFG["model"], "Precision"]) < 5e-4, \
        "Precision из матрицы не совпадает с таблицей результатов"
    assert abs(rec - results.loc[CFG["model"], "Recall"]) < 5e-4, \
        "Recall из матрицы не совпадает с таблицей результатов"
    print(f"✅ TEST 5 PASSED — TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"   Precision = TP/(TP+FP) = {prec:.4f} | Recall = TP/(TP+FN) = {rec:.4f}")

test_5_confusion()

In [ ]:
# 5.1. ROC- и PR-кривые для всех моделей (кроме baseline).
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.4))
for name, est in fitted.items():
    if name == "Baseline":
        continue
    try:
        prob = est.predict_proba(X_test)[:, 1]
    except AttributeError:
        continue
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax[0].plot(fpr, tpr, lw=1.8, label=f"{name} (AUC={roc_auc_score(y_test, prob):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, prob)
    ax[1].plot(rc, pr, lw=1.8, label=f"{name} (AP={average_precision_score(y_test, prob):.3f})")

ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="случайный выбор")
ax[0].set_title("ROC-кривая"); ax[0].set_xlabel("FPR = FP/(FP+TN)"); ax[0].set_ylabel("TPR = Recall")
ax[0].legend(fontsize=8, loc="lower right")

ax[1].axhline(y_test.mean(), ls="--", c="k", lw=1, label=f"базовый уровень = {y_test.mean():.3f}")
ax[1].set_title("Precision–Recall кривая"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[1].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

### 5.2. Порог принятия решения

Модель выдаёт **вероятность** $\hat{p} = P(y=1 \mid x)$. Решение «включить вентиляцию» получается сравнением с порогом $t$:

$$\hat{y} = \mathbb{1}[\hat{p} \ge t].$$

Порог `0.5` — не закон природы, а произвольное значение по умолчанию. Снижение $t$ поднимает Recall и опускает Precision; повышение $t$ — наоборот. Инженер выбирает $t$ исходя из **цены ошибок**, а не из симметрии.

Универсальный инструмент — метрика $F_\beta$: при $\beta = 2$ Recall весит вчетверо больше Precision, при $\beta = 0.5$ — наоборот.

In [ ]:
# TODO-7: СТУДЕНТ — ПОДБОР ПОРОГА ПОД ПРИОРИТЕТНУЮ МЕТРИКУ ВАРИАНТА
#
# 1. Возьмите вероятности модели варианта: proba = model_main.predict_proba(X_test)[:, 1]
# 2. Задайте beta: 2.0 если CFG['priority_metric'] == 'Recall', иначе 0.5
# 3. Переберите пороги np.linspace(0.05, 0.95, 91), для каждого посчитайте F_beta:
#       F_beta = (1+b^2) * P * R / (b^2 * P + R)   (при P+R == 0 считать 0)
# 4. Найдите best_threshold с максимумом F_beta, сохраните его в best_threshold.
# 5. Постройте график: по оси X — порог, по оси Y — Precision, Recall и F_beta (три линии),
#    вертикальной пунктирной линией отметьте best_threshold.
# 6. Выведите Precision/Recall/F1 при пороге 0.5 и при best_threshold — сравните.

best_threshold = None
# ВАШ КОД ЗДЕСЬ

In [ ]:
# ============ TEST 6 — порог подобран осмысленно ============
def test_6_threshold():
    assert best_threshold is not None, "Заполните best_threshold"
    assert 0.0 < best_threshold < 1.0, "Порог должен лежать в (0, 1)"
    proba = model_main.predict_proba(X_test)[:, 1]
    pred_t = (proba >= best_threshold).astype(int)
    pred_05 = (proba >= 0.5).astype(int)
    if CFG["priority_metric"] == "Recall":
        assert recall_score(y_test, pred_t, zero_division=0) >= \
               recall_score(y_test, pred_05, zero_division=0) - 1e-9, \
               "Приоритет варианта — Recall, а подобранный порог его не улучшил"
    else:
        assert precision_score(y_test, pred_t, zero_division=0) >= \
               precision_score(y_test, pred_05, zero_division=0) - 1e-9, \
               "Приоритет варианта — Precision, а подобранный порог его не улучшил"
    print(f"✅ TEST 6 PASSED — порог {best_threshold:.3f} согласован "
          f"с приоритетом «{CFG['priority_metric']}»")

test_6_threshold()

## Блок 6. Инженерно-педагогическая интерпретация — 2 балла

Ответьте развёрнуто (**не менее 5–7 предложений на вопрос**). Ответы «да/нет» и общие слова не засчитываются: требуется опора на **ваши** числа из таблицы результатов и матрицы ошибок.

---

**Вопрос 1. Что хуже для школы: ложное включение вентиляции (FP) или нераспознанный душный класс с детьми (FN)?**

Учтите: при 30 учениках в кабинете 60 м³ без вентиляции CO₂ достигает 1500 ppm примерно за 25–30 минут; при 1000+ ppm фиксируется падение скорости выполнения когнитивных задач. С другой стороны, каждое ложное срабатывание — это электроэнергия, износ оборудования, шум приточной установки во время урока и сквозняк зимой. Свяжите ответ с **приоритетной метрикой вашего варианта** и с фактическими числами FP и FN из вашей матрицы ошибок.

_ВАШ ОТВЕТ:_

---

**Вопрос 2. Почему accuracy — плохая метрика в этой задаче?**

Приведите числовой контрпример на **своих** данных: сравните accuracy baseline и accuracy лучшей модели, покажите, что разрыв в accuracy сильно меньше разрыва в F1.

_ВАШ ОТВЕТ:_

---

**Вопрос 3. Что произойдёт с моделью, обученной с признаком `date`, при переносе в школу другого региона или на летние каникулы?**

Опишите механизм отказа. Какие ещё «расписаниевые» признаки создали бы такую же проблему?

_ВАШ ОТВЕТ:_

---

**Вопрос 4. Какой порог и какой состав датчиков вы рекомендуете завучу — и почему?**

Сформулируйте как техническое решение: порог $t$, обоснование через $F_\beta$, требуемый минимальный комплект датчиков, что делать при отказе одного из них. Отдельно прокомментируйте роль `Light`: он даёт лучшие метрики, но физически связан с занятостью лишь косвенно (через привычку включать свет). Устойчива ли ваша модель к кабинету с проектором и задёрнутыми шторами?

_ВАШ ОТВЕТ:_

In [ ]:
# 6.1. Финальная сводка работы — приложите её скриншот/вывод к отчёту.
print("=" * 68)
print(f"  ЛР №1 · Вариант {VARIANT} · источник данных: {DATA_SOURCE}")
print("=" * 68)
print(f"  Сенсоры          : {CFG['sensor_set_id']} {CFG['sensors']}")
print(f"  Импутация        : {CFG['imputer']} | missing_rate = {CFG['missing_rate']:.3f}")
print(f"  Модель варианта  : {CFG['model']}")
print(f"  Приоритет        : {CFG['priority_metric']}")
print(f"  Порог решения    : {best_threshold:.3f}")
print(f"  Train / Test     : {len(X_train)} / {len(X_test)} наблюдений")
print("=" * 68)
print(results.to_string())
print("=" * 68)

# Выгрузка таблицы результатов (при желании — приложить к отчёту)
results.to_csv(f"lab01_results_variant_{VARIANT}.csv", encoding="utf-8-sig")
print(f"Сохранено: lab01_results_variant_{VARIANT}.csv")

## Чек-лист перед сдачей

- [ ] `VARIANT` соответствует моему номеру в ведомости
- [ ] Столбца `date` нет в `X_train.columns`
- [ ] Сплит хронологический, проверено `TEST 1`
- [ ] Импутер и скейлер находятся **внутри** `Pipeline`
- [ ] Обучены baseline + модель варианта + 2 модели сравнения
- [ ] Приведены Precision, Recall, F1 и confusion matrix для всех моделей
- [ ] Порог подобран под приоритетную метрику варианта
- [ ] У каждого графика есть заголовок, подписи осей и легенда
- [ ] Все шесть `TEST` выдают `✅ PASSED`
- [ ] Заполнены выводы 2.1, 2.2, 2.3, обоснование сплита и 4 вопроса блока 6
- [ ] `Runtime → Restart and run all` проходит без ошибок
- [ ] Ноутбук выгружен **с сохранёнными выводами ячеек**

**Сдача:** файл `lab_01_<Фамилия>_v<номер>.ipynb` в LMS курса.